In [ ]:
import plotly.express as ex
import polars as pl

from desdeo.emo import DesirableRangesOptions, emo_constructor, nsga3_options
from desdeo.emo.options.generator import ArchiveGeneratorOptions
from desdeo.problem import Problem
from desdeo.problem.testproblems import rocket_injector_design, dtlz2
from desdeo.tools.score_bands import KMeansOptions, SCOREBandsConfig, plot_score, score_json, DimensionClusterOptions

In [ ]:
def solve(
    problem: Problem,
    solutions: pl.DataFrame | None=None,
    outputs: pl.DataFrame | None = None,
    preference: dict[str, dict[str, float]] | None = None,
    pop_size: int = 100,
    ):
    opts = nsga3_options()
    if preference is not None:
        dr_opts = DesirableRangesOptions(
            aspiration_levels=preference["aspiration_levels"],
            reservation_levels=preference["reservation_levels"],
            method="DF transformation",
            desirability_levels=(0.999, 0.001)
            )
        opts.preference = dr_opts
    opts.template.generator.n_points = pop_size
    opts.template.selection.reference_vector_options.number_of_vectors = pop_size
    opts.template.selection.invert_reference_vectors = True
    if solutions is not None and outputs is not None:
        opts.template.generator = ArchiveGeneratorOptions(
            solutions=solutions, outputs=outputs
        )
    solver, extras = emo_constructor(problem=problem, emo_options=opts)
    res =solver()
    #return extras.archive.results
    return res, extras.archive.results

def band_to_pref(score_data, band_index, problem: Problem) -> dict[str, dict[str, float]]:
    band_data = score_data[-1].bands[band_index]
    return {
        "aspiration_levels": {obj.symbol: band_data[obj.symbol][obj.maximize] for obj in problem.objectives},
        "reservation_levels": {obj.symbol: band_data[obj.symbol][not obj.maximize] for obj in problem.objectives}
    }

def preference_filter(solutions: pl.DataFrame, preference: dict[str, dict[str, float]| None], problem: Problem, num_solutions: int | None=None) -> pl.DataFrame:
    """Filter the solutions based on the preference."""
    if preference is None:
        return solutions
    # Example filter: only keep solutions that are between the reservation and aspiration levels
    reservation_levels = preference["reservation_levels"]
    aspiration_levels = preference["aspiration_levels"]
    for obj in problem.objectives:
        if obj.maximize:
            solutions = solutions.filter(
                (pl.col(obj.symbol) >= reservation_levels[obj.symbol])# & 
                #(pl.col(obj.symbol) <= aspiration_levels[obj.symbol])
            )
        else:
            solutions = solutions.filter(
                (pl.col(obj.symbol) <= reservation_levels[obj.symbol])# & 
                #(pl.col(obj.symbol) >= aspiration_levels[obj.symbol])
            )
    # If more than a 1000 solutions, choose 1000 random solutions
    if num_solutions is not None and len(solutions) > num_solutions:
        solutions = solutions.sample(n=num_solutions, with_replacement=False)
    return solutions

In [ ]:
rocket_problem = rocket_injector_design(original_version=False)
#problem = dtlz2(10, 3)
with open("metall.json") as f:
    problem_data = f.read()
metall_problem = Problem.model_validate_json(problem_data)

problem = metall_problem

ideal = problem.get_ideal_point()
nadir = problem.get_nadir_point()
columns = list(ideal.keys())

results = []
archives = []
preferences = []
r, a = solve(problem)
results.append(pl.concat([r.optimal_variables, r.optimal_outputs[columns]], how="horizontal"))
archives.append(pl.concat([a.optimal_variables, a.optimal_outputs[columns]], how="horizontal"))

In [ ]:
score_data = []
score_config = SCOREBandsConfig()
score_config.clustering_algorithm = KMeansOptions(n_clusters=5)
#score_config.clustering_algorithm = DimensionClusterOptions(dimension_name="TF_max", n_clusters=5)
score_config.distance_parameter = 0.3

In [ ]:
score_data.append(score_json(
        data=archives[-1][columns],
        options=score_config))

score_config.scales = score_data[0].options.scales  # Set scales based on first run
score_config.axis_positions = score_data[0].axis_positions  # Set axis positions based on first run

In [ ]:
iteration = 0

plot_score(data=archives[iteration][columns], result=score_data[iteration])

In [ ]:
def plot_3d(datalist, preferences, problem):
    fig = ex.scatter_3d()
    for i, result in enumerate(datalist):
        fig.add_scatter3d(
            x=result[columns[0]],
            y=result[columns[1]],
            z=result[columns[2]],
            mode="markers",
        marker=dict(size=4, opacity=0.7),
        name=f"Preferred optimal solutions {i+1}"
        )
        if i > 0:
            # Make box from aspiration and reservation levels
            aspiration = preferences[i-1]["aspiration_levels"]
            reservation = preferences[i-1]["reservation_levels"]
            fig.add_scatter3d(
                x=[reservation[columns[0]], aspiration[columns[0]], aspiration[columns[0]], reservation[columns[0]], reservation[columns[0]], reservation[columns[0]], aspiration[columns[0]], aspiration[columns[0]], reservation[columns[0]]],
                y=[reservation[columns[1]], reservation[columns[1]], aspiration[columns[1]], aspiration[columns[1]], reservation[columns[1]], reservation[columns[1]], aspiration[columns[1]], aspiration[columns[1]], reservation[columns[1]]],
                z=[reservation[columns[2]], reservation[columns[2]], reservation[columns[2]], reservation[columns[2]], aspiration[columns[2]], aspiration[columns[2]], aspiration[columns[2]], aspiration[columns[2]], reservation[columns[2]]],
                mode="lines",
                line=dict(color="red", width=4),
                name=f"Preference box {i}"
            )
            # all solutions inside the box
            solutions = datalist[0]
            reservation_levels = preferences[i-1]["reservation_levels"]
            aspiration_levels = preferences[i-1]["aspiration_levels"]
            for obj in problem.objectives:
                if obj.maximize:
                    solutions = solutions.filter(
                        (pl.col(obj.symbol) >= reservation_levels[obj.symbol]) & 
                        (pl.col(obj.symbol) <= aspiration_levels[obj.symbol])
                    )
                else:
                    solutions = solutions.filter(
                        (pl.col(obj.symbol) <= reservation_levels[obj.symbol]) & 
                        (pl.col(obj.symbol) >= aspiration_levels[obj.symbol])
                    )
            fig.add_scatter3d(
                x=solutions[columns[0]],
                y=solutions[columns[1]],
                z=solutions[columns[2]],
                mode="markers",
                marker=dict(size=4, opacity=0.7),
                name=f"Solutions better than reservation {i}"
            )
    # Set axis labels
    fig.update_layout(
        scene = dict(
            xaxis_title=columns[0],
            yaxis_title=columns[1],
            zaxis_title=columns[2]
        )
    )
    return fig
plot_3d(archives, preferences, problem).show()

In [ ]:
preference = band_to_pref(score_data, 3, problem=problem)
preference

In [ ]:
preferences.append(preference)

seed_gen = preference_filter(pl.concat(archives), preference=preference, problem=problem, num_solutions=300)
var_names = [var.symbol for var in problem.get_flattened_variables()]
output_names = [name for name in results[0].columns if name not in var_names]
seed_sols = seed_gen[var_names]
seed_outs = seed_gen[output_names]

r, a = solve(problem, preference=preferences[-1], solutions=seed_sols, outputs=seed_outs)

results.append(preference_filter(
    pl.concat([r.optimal_variables, r.optimal_outputs[columns]], how="horizontal"), preference=preferences[-1], problem=problem))
archives.append(preference_filter(
    pl.concat([a.optimal_variables, a.optimal_outputs[columns]], how="horizontal"), preference=preferences[-1], problem=problem))